# Brightness under each technique: 3D true vs 3D stitched, region by region

Section 7 of `segmentation_method_comparison.ipynb` walks every FOV in a
region under both techniques and scores them on *mask shape* — how far masks
reach through z, how steadily they change size. This notebook walks the same
FOVs, region by region, and asks what the mask geometry cannot: **do the voxels
a technique claims actually look like signal?**

Every voxel contributes one `(brightness, is_masked)` pair — each pixel column
`(y, x)` vectorised across z, so a technique's decision is scored plane by plane
rather than object by object. A technique picking up nuclei puts bright voxels
inside its masks and leaves dim ones outside. One that over-grows its masks
drags background in, and its masked distribution slides left toward the
unmasked one.

## The figure

One figure per FOV, two panels:

* **left** — voxels *in* a mask, one curve per technique
* **right** — voxels *not* in a mask, one curve per technique

This is the opposite split from the single-run figure in
`segmentation_z_brightness_claude.py`, where the two panels were masked vs
unmasked for one technique. Here each panel holds one of those groups and the
*techniques* are overlaid inside it, because the comparison being made is
between techniques — and a difference is far easier to read as two curves on one
axes than as two panels side by side.

Each curve is normalised within its own (technique, panel) group. That matters:
masked voxels are a few percent of a volume and unmasked voxels are the rest, so
raw counts would only ever show that one group is enormous.

Both techniques segment the **same DAPI stack**, so the stack is read once per
FOV, the bins are built from it once, and both mask volumes are histogrammed
onto them — two distributions binned differently cannot be laid over each other.
Across FOVs the bins do differ, so `SHARE_X` pins every figure in a region to
one axis and the stack of them reads down as well as across.

## Reading it

`auc` is the headline: **P(a random masked voxel is brighter than a random
unmasked one)** within that run. 0.5 says brightness carries no information about
what the technique masked; 1.0 says every masked voxel outshines every unmasked
one. It is the one number that survives the two groups being wildly different
sizes, so it compares techniques and FOVs directly — but read it next to
`frac_masked`, because a technique can post a high AUC by masking only the very
brightest voxels and skipping most of the nuclei.

`segmentation_brightness_helpers.py` is only the comparison layer: the
measurement comes from `segmentation_z_brightness_claude.py` and the region walk
and palette from `segmentation_z_helpers.py`, so a technique keeps its colour
across both notebooks.

In [ ]:
from pathlib import Path

import pandas as pd

import segmentation_z_helpers as h
import segmentation_brightness_helpers as bh

h.apply_theme()   # same figure style as the other notebook
pd.set_option("display.width", 140)

### Config

`TECHNIQUE_ROOTS` points at the directory *above* the region under each
technique tree, exactly as in section 7. `DAPI_PATCHES` is the matching patches
directory — each region block appends its own region name to it, since that is
the image both techniques segmented.

In [ ]:
DATA_ROOT = Path("data")

FAMILY = "cpdino"
PREPROCESSING = "decon"
MODEL = "VePo"

TECHNIQUE_ROOTS = {
    "3D true":     DATA_ROOT / "segmentations_3d_true"     / FAMILY / PREPROCESSING / MODEL,
    "3D stitched": DATA_ROOT / "segmentations_3d_stitched" / FAMILY / PREPROCESSING / MODEL,
}
DAPI_PATCHES = DATA_ROOT / "patches" / MODEL

N_DISPLAY_BINS = 60
CLIP_QUANTILE = 0.999   # x axis stops at this quantile of the pooled brightness
SHARE_X = True          # one brightness axis across every FOV in the region
SHARE_Y = True          # one height across both panels; False lets each autoscale
LOG_Y = False           # worth flipping when a peak flattens the tails

METRIC_COLUMNS = ["frac_masked", "median_masked", "median_unmasked",
                  "auc", "point_biserial_r"]

print("available:", ", ".join(h.available_regions(TECHNIQUE_ROOTS)))

### Region UCI-2424

In [ ]:
# --- 1a. every FOV in the region, under both techniques ----------------------
SCAN_REGION = "region_UCI-2424"

found = h.find_region_fovs(TECHNIQUE_ROOTS, SCAN_REGION)
print(f"\n{found['fov'].nunique()} paired FOV(s) in {SCAN_REGION}\n")

runs = bh.scan_region_brightness(found, DAPI_PATCHES / SCAN_REGION)
metrics = bh.brightness_metrics(runs)

metrics[METRIC_COLUMNS].round(3)

In [ ]:
# --- 1b. one figure per FOV ---------------------------------------------------
figures = bh.plot_region_brightness(
    runs, metrics=metrics,
    share_x=SHARE_X, share_y=SHARE_Y,
    n_bins=N_DISPLAY_BINS, clip_quantile=CLIP_QUANTILE, log_y=LOG_Y,
)

### Region UCI-4723

In [ ]:
# --- 2a. every FOV in the region, under both techniques ----------------------
SCAN_REGION = "region_UCI-4723"

found = h.find_region_fovs(TECHNIQUE_ROOTS, SCAN_REGION)
print(f"\n{found['fov'].nunique()} paired FOV(s) in {SCAN_REGION}\n")

runs = bh.scan_region_brightness(found, DAPI_PATCHES / SCAN_REGION)
metrics = bh.brightness_metrics(runs)

metrics[METRIC_COLUMNS].round(3)

In [ ]:
# --- 2b. one figure per FOV ---------------------------------------------------
figures = bh.plot_region_brightness(
    runs, metrics=metrics,
    share_x=SHARE_X, share_y=SHARE_Y,
    n_bins=N_DISPLAY_BINS, clip_quantile=CLIP_QUANTILE, log_y=LOG_Y,
)

### Region UCI-5224

In [ ]:
# --- 3a. every FOV in the region, under both techniques ----------------------
SCAN_REGION = "region_UCI-5224"

found = h.find_region_fovs(TECHNIQUE_ROOTS, SCAN_REGION)
print(f"\n{found['fov'].nunique()} paired FOV(s) in {SCAN_REGION}\n")

runs = bh.scan_region_brightness(found, DAPI_PATCHES / SCAN_REGION)
metrics = bh.brightness_metrics(runs)

metrics[METRIC_COLUMNS].round(3)

In [ ]:
# --- 3b. one figure per FOV ---------------------------------------------------
figures = bh.plot_region_brightness(
    runs, metrics=metrics,
    share_x=SHARE_X, share_y=SHARE_Y,
    n_bins=N_DISPLAY_BINS, clip_quantile=CLIP_QUANTILE, log_y=LOG_Y,
)

### Region UWA-7648

In [ ]:
# --- 4a. every FOV in the region, under both techniques ----------------------
SCAN_REGION = "region_UWA-7648"

found = h.find_region_fovs(TECHNIQUE_ROOTS, SCAN_REGION)
print(f"\n{found['fov'].nunique()} paired FOV(s) in {SCAN_REGION}\n")

runs = bh.scan_region_brightness(found, DAPI_PATCHES / SCAN_REGION)
metrics = bh.brightness_metrics(runs)

metrics[METRIC_COLUMNS].round(3)

In [ ]:
# --- 4b. one figure per FOV ---------------------------------------------------
figures = bh.plot_region_brightness(
    runs, metrics=metrics,
    share_x=SHARE_X, share_y=SHARE_Y,
    n_bins=N_DISPLAY_BINS, clip_quantile=CLIP_QUANTILE, log_y=LOG_Y,
)

### Looking at one FOV in napari

Set `NAPARI_REGION` and `NAPARI_FOV` in the cell below — it reads that pair on
its own, so it does not matter which region block above you ran last, or whether
you ran one at all. Name a FOV that is not paired in that region and it says
which ones are.

Per technique, two derived volumes on top of the DAPI and its masks:

* **bright, unmasked** — outside every mask yet brighter than the median masked
  voxel. Signal that technique left behind.
* **dim, masked** — inside a mask yet dimmer than the median unmasked voxel.
  Background it pulled in.

Thresholds are per technique, since each one's own distribution is what defines
bright and dim for it — which is why the stitched thresholds come out lower.
Both derived layers start hidden.

In [ ]:
# --- 5a. pick a region and a FOV, then build the layers (safe to run headless) -
NAPARI_REGION = "region_UCI-2424"
NAPARI_FOV    = "fov_07"

focus_dapi, focus_masks, focus_edges, focus_counts = bh.load_fov(
    TECHNIQUE_ROOTS, DAPI_PATCHES, NAPARI_REGION, NAPARI_FOV)

focus_layers = bh.build_overlap_layers(
    focus_dapi, focus_masks, focus_edges, focus_counts)

for method, entry in focus_layers.items():
    print(f"{NAPARI_REGION}  {NAPARI_FOV}  {method:<12} "
          f"{int(entry['bright_unmasked'].sum()):>9,} bright-but-unmasked "
          f"(>{entry['bright_thr']:,.0f})   "
          f"{int(entry['dim_masked'].sum()):>8,} dim-but-masked "
          f"(<{entry['dim_thr']:,.0f})")

In [ ]:
# --- 5b. two viewers, one per technique ---------------------------------------
# Blocks the kernel until BOTH windows are closed.
viewers = bh.launch_viewer(focus_dapi, focus_masks, focus_layers,
                           title=f"{NAPARI_REGION} {NAPARI_FOV}")